# Diagnostica — Fonti, Cataloghi, Dataset

Vista unificata dello stato del Source Observatory:
- **Fonti**: radar health, quante monitorate vs inventariate
- **Cataloghi**: item count, delta, segnali di drift
- **Dataset**: copertura source_check, intake score, reachability, granularità
- **Copertura**: % inventario processato, gap

**Data**: 2026-06-08

In [1]:
import json

from lab_connectors.duckdb import gcs_connect

# ── Path ──
RADAR_PATH = "../data/radar/radar_summary.json"
SIGNALS_PATH = "../data/catalog/catalog_signals.json"
S3_INV = "s3://dataciviclab-clean/catalog_inventory/catalog_inventory_latest.parquet"
S3_SCR = "s3://dataciviclab-clean/catalog_inventory/source-check/source_check_results.parquet"

# ── Carica dati locali (git) ──
with open(RADAR_PATH) as f:
    radar = json.load(f)
with open(SIGNALS_PATH) as f:
    signals = json.load(f)

# ── Carica dati da S3 ──
with gcs_connect(S3_INV) as con:
    df_inv = con.execute("SELECT * FROM read_parquet(?)", [S3_INV]).fetchdf()
with gcs_connect(S3_SCR) as con:
    df_scr = con.execute("SELECT * FROM read_parquet(?)", [S3_SCR]).fetchdf()

print(f"Radar: {radar['sources_total']} fonti")
print(f"Inventory: {len(df_inv)} rows, {df_inv['source_id'].nunique()} fonti")
print(f"Source-check: {len(df_scr)} rows, {df_scr['source_id'].nunique()} fonti")
print(f"Signals: {signals['sources_checked']} fonti controllate")

Radar: 30 fonti
Inventory: 15007 rows, 26 fonti
Source-check: 9771 rows, 24 fonti
Signals: 26 fonti controllate


---
## 1. Fonti — Radar Health

In [2]:
print("═" * 60)
print("  RADAR HEALTH")
print("═" * 60)
print(f"\nGREEN:  {radar['status_counts'].get('GREEN', 0)}")
print(f"YELLOW: {radar['status_counts'].get('YELLOW', 0)}")
print(f"RED:    {radar['status_counts'].get('RED', 0)}")
print(f"Persistent RED: {radar['persistent_red']}\n")

print("Fonti non VERDI:")
for s in radar["sources"]:
    if s["status"] != "GREEN":
        print(f"  🔴 {s['id']:<25s} {s['status']:<8s} {s.get('note', '')[:80]}")

# Fonti radar MA non in inventory
radar_ids = {s["id"] for s in radar["sources"]}
inv_ids = set(df_inv["source_id"].unique()) if not df_inv.empty else set()
not_inventoried = radar_ids - inv_ids
if not_inventoried:
    print(f"\nFonti non inventariate ({len(not_inventoried)}):")
    for sid in sorted(not_inventoried):
        s = next((s for s in radar["sources"] if s["id"] == sid), {})
        print(f"  {sid:<25s} protocol={s.get('protocol', '?'):<10s}")

inventoried_not_radar = inv_ids - radar_ids
if inventoried_not_radar:
    print(f"\nIn inventory ma non in radar ({len(inventoried_not_radar)}): {inventoried_not_radar}")

════════════════════════════════════════════════════════════
  RADAR HEALTH
════════════════════════════════════════════════════════════

GREEN:  30
YELLOW: 0
RED:    0
Persistent RED: 0

Fonti non VERDI:

Fonti non inventariate (4):
  dait                      protocol=html      
  inail_opendata            protocol=aem       
  noipa_sparql              protocol=sparql    
  terna_opendata            protocol=rest      


---
## 2. Cataloghi — Item count e segnali

In [3]:
print("═" * 60)
print("  CATALOGHI — Item per fonte")
print("═" * 60)

src_counts = df_inv.groupby(["source_id", "protocol"]).size().reset_index(name="items")
src_counts = src_counts.sort_values("items", ascending=False)
print(src_counts.to_string(index=False))

print(f"\nTotale item inventario: {len(df_inv)}")

════════════════════════════════════════════════════════════
  CATALOGHI — Item per fonte
════════════════════════════════════════════════════════════
            source_id protocol  items
           istat_sdmx     sdmx   4871
             openbdap     ckan   3817
                 inps     ckan   2323
          opencivitas     html    703
         opencoesione     ckan    561
               openga     ckan    436
         mim_opendata     html    372
            mimit_rna     ckan    370
          unioncamere     ckan    352
         dati_cultura   sparql    213
            mef_irpef     html    147
          dati_camera   sparql    104
          dati_senato   sparql     98
      lavoro_opendata     ckan     83
                 anac     ckan     70
         mit_opendata     ckan     70
            mur_ustat     ckan     69
    ispra_linked_data   sparql     67
                 agcm     ckan     53
     ministero_salute     ckan     51
                 aifa     html     42
             

In [4]:
print("═" * 60)
print("  SEGNALI DI DRIFT")
print("═" * 60)

for sig in signals["signals"]:
    if sig["signal_type"] != "no signal":
        print(f"\n{sig['source']:<22s} {sig['signal_type']:<20s}")
        print(f"    {sig.get('detail', '')[:150]}")
        print(f"    azione: {sig['suggested_action']}")

════════════════════════════════════════════════════════════
  SEGNALI DI DRIFT
════════════════════════════════════════════════════════════

mim_opendata           csv_magnet          
    1116 link data (CSV 372, JSON 372, XML 372), years 2015-2026 — top prefixes: INFANZIA=192, ALUCORSO=120, SCUANAGR=66, SCUANAAU=66, ALUITAST=60
    azione: catalog-watch-ready

ispra_linked_data      inventory change    
    67 item (sparql_query), delta +1 rispetto al run precedente (66).
    azione: verificare se variazione attesa; avviare inventory-triage se nuovi dataset

mef_irpef              csv_magnet          
    HTTPSConnectionPool(host='www1.finanze.gov.it', port=443): Max retries exceeded with url: /finanze/analisi_stat/public/index.php?opendata=yes (Caused 
    azione: verificare raggiungibilità del portale

opencivitas            csv_magnet          
    748 link data (ZIP 748), years 2010-2025 — top prefixes: 2010=90, 2013=72, Metadati=64, 2022=63, 2018=59
    azione: catalog-watch-re

---
## 3. Dataset — Source-check coverage

In [5]:
print("═" * 60)
print("  COPERTURA SOURCE-CHECK")
print("═" * 60)

# Merge inventory + source_check
inv_count = df_inv.groupby("source_id").size().reset_index(name="inventory_items")
scr_count = df_scr.groupby("source_id").size().reset_index(name="scored_items")
coverage = inv_count.merge(scr_count, on="source_id", how="outer").fillna(0)
coverage["pct"] = (
    coverage["scored_items"] / coverage["inventory_items"].clip(lower=1) * 100
).round(1)
coverage = coverage.astype({"inventory_items": int, "scored_items": int})
coverage = coverage.sort_values("inventory_items", ascending=False)

print(f"{'Fonte':<22s} {'Inventory':>10s} {'Scored':>8s} {'Coperto':>8s}")
print("-" * 50)
for _, r in coverage.iterrows():
    print(
        f"{r['source_id']:<22s} {r['inventory_items']:>10d} {r['scored_items']:>8.0f} {r['pct']:>7.1f}%"
    )
print(f"\nTotale inventory: {coverage['inventory_items'].sum():.0f}")
print(f"Totale scored:    {coverage['scored_items'].sum():.0f}")
print(
    f"Copertura media:  {coverage['scored_items'].sum() / coverage['inventory_items'].sum() * 100:.1f}%"
)

════════════════════════════════════════════════════════════
  COPERTURA SOURCE-CHECK
════════════════════════════════════════════════════════════
Fonte                   Inventory   Scored  Coperto
--------------------------------------------------
istat_sdmx                   4871     3584    73.6%
openbdap                     3817     1918    50.2%
inps                         2323     1426    61.4%
opencivitas                   703      384    54.6%
opencoesione                  561      504    89.8%
openga                        436      364    83.5%
mim_opendata                  372      387   104.0%
mimit_rna                     370       85    23.0%
unioncamere                   352      349    99.1%
dati_cultura                  213      213   100.0%
mef_irpef                     147       80    54.4%
dati_camera                   104      104   100.0%
dati_senato                    98        0     0.0%
lavoro_opendata                83       83   100.0%
anac                  

In [6]:
print("═" * 60)
print("  QUALITA' DATASET")
print("═" * 60)

total = len(df_scr)
print(f"Item totali:            {total:>6}")
print(
    f"Candidati intake:       {df_scr['intake_candidate'].sum():>6} ({df_scr['intake_candidate'].sum() / total * 100:.0f}%)"
)
print(
    f"Needs review:           {df_scr['needs_review'].sum():>6} ({df_scr['needs_review'].sum() / total * 100:.0f}%)"
)
print(
    f"Reachable:              {df_scr['reachable'].sum():>6} ({df_scr['reachable'].sum() / total * 100:.0f}%)"
)
print(
    f"Con colonne profilate:  {df_scr['columns'].notna().sum():>6} ({df_scr['columns'].notna().sum() / total * 100:.0f}%)"
)
print(
    f"Con join_keys:          {df_scr['join_keys'].notna().sum():>6} ({df_scr['join_keys'].notna().sum() / total * 100:.0f}%)"
)
print(f"Intake score medio:     {df_scr['intake_score'].mean():6.1f}")
if "joinability_score" in df_scr.columns:
    print(f"Joinability score medio: {df_scr['joinability_score'].mean():6.1f}")

════════════════════════════════════════════════════════════
  QUALITA' DATASET
════════════════════════════════════════════════════════════
Item totali:              9771
Candidati intake:         1271 (13%)
Needs review:             7946 (81%)
Reachable:                2853 (29%)
Con colonne profilate:    2118 (22%)
Con join_keys:            1009 (10%)
Intake score medio:       26.0
Joinability score medio:    5.1


In [7]:
print("═" * 60)
print("  GRANULARITA'")
print("═" * 60)
gran = df_scr["granularity"].value_counts()
for g, c in gran.items():
    print(f"  {g:<20s} {c:>5} ({c / total * 100:.0f}%)")

print("\n  FORMATI")
fmt = df_scr["resource_format"].value_counts()
for f, c in fmt.head(8).items():
    print(f"  {str(f):<10s} {c:>5} ({c / total * 100:.0f}%)")

════════════════════════════════════════════════════════════
  GRANULARITA'
════════════════════════════════════════════════════════════
  non_determinato       6713 (69%)
  regione               1607 (16%)
  comune                 446 (5%)
  provincia              372 (4%)
  nazionale              345 (4%)
  europeo                 99 (1%)
  mondiale                 2 (0%)

  FORMATI
  CSV         4455 (46%)
  SDMX        3584 (37%)
  XLS          640 (7%)
  ZIP          437 (4%)
               317 (3%)
  XML          130 (1%)
  JSON          12 (0%)
  PDF            7 (0%)


---
## 4. Sintesi

Cose da fare:
- Fonti non inventariate → valutare se aggiungere a catalog-watch
- Fonti con segnali di drift → verificare
- Source_check da completare su fonti a bassa copertura
- Dataset con intake_score alto ma senza join_keys → potenziali falsi negativi

In [8]:
print("═" * 60)
print("  COSE DA FARE")
print("═" * 60)

# 1. Fonti non inventariate
if not_inventoried:
    print(f"\n📋 Fonti da inventariare: {', '.join(sorted(not_inventoried))}")

# 2. Fonti con segnali
signals_active = [s for s in signals["signals"] if s["signal_type"] != "no signal"]
if signals_active:
    print(f"\n📋 Fonti con segnali attivi ({len(signals_active)}):")
    for s in signals_active:
        print(
            f"     {s['source']:<22s} {s['signal_type']:<20s} {s.get('suggested_action', '')[:60]}"
        )

# 3. Fonti con bassa copertura source-check
low_cov = coverage[coverage["pct"] < 50].sort_values("pct")
if not low_cov.empty:
    print(f"\n📋 Fonti con copertura < 50% ({len(low_cov)}):")
    for _, r in low_cov.iterrows():
        print(
            f"     {r['source_id']:<22s} {r['pct']:.0f}% ({r['scored_items']:.0f}/{r['inventory_items']:.0f})"
        )

# 4. Intake alto senza join_keys
high_intake_no_keys = df_scr[(df_scr["intake_score"] >= 50) & (df_scr["join_keys"].isna())].copy()
if len(high_intake_no_keys) > 0:
    print(f"\n📋 Dataset con intake >= 50 MA senza join_keys ({len(high_intake_no_keys)}):")
    top = high_intake_no_keys.sort_values("intake_score", ascending=False).head(5)
    for _, r in top.iterrows():
        print(
            f"     {r['source_id']:<15s} score={r['intake_score']:.0f} {str(r.get('title', ''))[:50]}"
        )

════════════════════════════════════════════════════════════
  COSE DA FARE
════════════════════════════════════════════════════════════

📋 Fonti da inventariare: dait, inail_opendata, noipa_sparql, terna_opendata

📋 Fonti con segnali attivi (7):
     mim_opendata           csv_magnet           catalog-watch-ready
     ispra_linked_data      inventory change     verificare se variazione attesa; avviare inventory-triage se
     mef_irpef              csv_magnet           verificare raggiungibilità del portale
     opencivitas            csv_magnet           catalog-watch-ready
     aifa                   csv_magnet           catalog-watch-ready
     giustizia_statistiche  csv_magnet           low signal
     cortecostituzionale    csv_magnet           catalog-watch-ready

📋 Fonti con copertura < 50% (8):
     dati_senato            0% (0/98)
     ispra_linked_data      0% (0/67)
     agcm                   8% (4/53)
     anac                   19% (13/70)
     agid                   20%

---
## 5. Note

- I dati radar e signals vengono dal repository git (ultimo commit).
- I dati inventory e source_check vengono da S3 (ultimo run CI).
- La diagnostica non modifica nulla — è una vista sola.